## DP2 Tables
Here we'll take a look at how to utilise the DP2 catalogues. 

### Visit table
Includes metadata of each exposure taken as part of DP2.  This includes the filter, exposure time, MJD of the exposure, etc.  It can be joined to other tables using the 'visit' column.

In [ ]:
import pandas as pd
visit = pd.read_parquet('data/Visit.parquet')
visit

### DiaObject table
This table containes a row for each difference image analysis (DIA) object.  In other words, multiple detections of the same object are grouped into one row along with some basic summary data (e.g. number of detections in each filter, peak flux, fade rate, etc.).

In [ ]:
diaobject = pd.read_parquet('data/DiaObject.parquet')
diaobject

### ForcedSourceOnDiaObject Table
This table contains all the photometry for each DIA object. It does not contain MJDs, filter or coordinates so this will need to be joined from the Visit and DiaObject tables. 
It is huge so I would caution against reading it entirely into memory unless you have an incredibly expensive computer.  Instead, we'll use `pyarrow` which can be enables querying parquet files without fully reading them into memeory

In [ ]:
import pyarrow.dataset as ds

def get_lightcurve(obj):
    dataset = ds.dataset('data/ForcedSourceOnDiaObject.parquet', format='parquet')
    forced_lc = dataset.to_table(
        filter=ds.field('diaObjectId') == obj
    ).to_pandas()

    # Add MJDs and filters
    forced_lc = pd.merge(
        forced_lc,
        visit[['visit', 'expMidptMJD']],
        on='visit',
        how='left'
    )
    return forced_lc

obj = 759230302445896656 # diaObjectId of interest
forced_lc = get_lightcurve(obj)

## Plotting light curves
### Flux
The columns of we're interested in here are `psfFlux` which is the flux density in nJy as measured via a PSF fit on the *science* image or psfDiffFlux which is measured on the *difference* image.
Beware of using *psfDiffFlux* because templates are generated using a subset of science images so even genuine transients will often have negative flux values.

In [ ]:
import matplotlib.pyplot as plt

FILTER_COLORS = {
    'u': 'violet',
    'g': 'dodgerblue',
    'r': 'seagreen',
    'i': 'goldenrod',
    'z': 'orangered',
    'y': 'maroon',
}

def plot_flux_lc(forced_lc, diff=False):
    flux_key = 'psfFlux' if not diff else 'diffFlux'

    for band, color in FILTER_COLORS.items():
        band_lc = forced_lc[(forced_lc['band'] == band)]
        if len(band_lc) == 0:
            continue
        plt.errorbar(
            band_lc['expMidptMJD'],
            band_lc[flux_key] * 1e-3,
            yerr=band_lc[flux_key + 'Err'] * 1e-3,
            fmt='o',
            color=color,
            label=band,
            capsize=3,
            markeredgecolor='k',
            markerfacecolor=color,
            alpha=0.7,
        )
    plt.xlabel('MJD')
    plt.ylabel(r'Flux density ($\mu$Jy)')
    plt.axhline(0, color='k', linestyle='--', alpha=0.5, zorder=-1)
    plt.legend()
    plt.show()

plot_flux_lc(forced_lc)

### Magnitudes
Here is the same light curve for magnitude enjoyers. 

In [ ]:
import astropy.units as u
import numpy as np

def plot_mag_lc(forced_lc):
    for band, color in FILTER_COLORS.items():
        band_lc = forced_lc[(forced_lc['band'] == band) & (forced_lc['mag_err'] < 0.5)]
        if len(band_lc) == 0:
            continue
        plt.errorbar(
            band_lc['expMidptMJD'],
            band_lc['mag'],
            yerr=band_lc['mag_err'],
            fmt='o',
            color=color,
            label=band,
            capsize=3,
            markeredgecolor='k',
            markerfacecolor=color,
            alpha=0.7,
        )
    plt.xlabel('MJD')
    plt.ylabel(r'AB Magnitude')
    plt.gca().invert_yaxis()
    plt.legend()
    plt.show()

forced_lc['mag'] = (np.array(forced_lc['psfFlux']) * u.nJy).to(u.ABmag).value
forced_lc['mag_err'] = 2.5 * np.log10(np.e) * np.abs(forced_lc['psfFluxErr']/forced_lc['psfFlux']) # approximate

plot_mag_lc(forced_lc)


### Correcting for extinction
To make statements about the color and luminosity of our transient of interest, we need to correct for Galactic extinction.

In [ ]:
# This fixes the broken fetch function in dustmaps
import requests

old_get = requests.get

def get_with_user_agent(*args, **kwargs):
    kwargs.setdefault("headers", {})
    kwargs["headers"]["User-Agent"] = "Mozilla/5.0"
    return old_get(*args, **kwargs)

requests.get = get_with_user_agent

In [ ]:
import extinction
from dustmaps.sfd import SFDQuery
from astropy.coordinates import SkyCoord
from dustmaps.sfd import fetch
fetch() # Download the SFD dust maps if they are not already available

# Taken from https://lsstcam.lsst.io/
FILTER_WAVELS = {
    'u': 3724,
    'g': 4807,
    'r': 6221,
    'i': 7559,
    'z': 8680,
    'y': 9753
}

def correct_extinction(mag, bands, coords):
    sfd = SFDQuery()
    av = 3.1 * sfd(coords)
    wavels = np.array(bands.map(FILTER_WAVELS))
    mag_ext = extinction.fm07(wavels, av)
    return mag - mag_ext

# Get coordinates of the source from the DiaObject table
ra, dec = diaobject[diaobject['diaObjectId'] == obj][['ra', 'dec']].values[0]
coords = SkyCoord(ra=ra*u.deg, dec=dec*u.deg) 

forced_lc['mag'] = correct_extinction(
    forced_lc['mag'],
    forced_lc['band'],
    coords,
)
plot_mag_lc(forced_lc)

## Filtering
Writing some basic filters to identify fast transient candidates.
### Duration
To filter on the duration of a given transient, we'll need to do a table join and aggregate the data from each object.  Again we can't read the entire catalogue into memory so we will have to do this in batches.

In [ ]:
forced_path = 'data/ForcedSourceOnDiaObject.parquet'
obj_ids = list(diaobject['diaObjectId'])
batch_size = 100_000

dataset = ds.dataset(forced_path, format='parquet')
needed_cols = ['diaObjectId', 'expMidptMJD', 'psfFlux', 'psfFluxErr']
visit_mjd = visit.set_index('visit')['expMidptMJD']  # only needed if merging
obj_id_set = set(diaobject['diaObjectId'])  # isin() against a set/array works fine

# Summarize observation MJDs for each diaObjectId (all observations)
partials = []
forced_path = 'data/ForcedSourceOnDiaObject.parquet'
obj_ids = list(diaobject['diaObjectId'])
batch_size = 100_000

dataset = ds.dataset(forced_path, format='parquet')

# expMidptMJD belongs to the Visit table, so query visit instead.
needed_cols = ['diaObjectId', 'visit', 'psfFlux', 'psfFluxErr']
visit_mjd = visit.set_index('visit')['expMidptMJD']
obj_id_set = set(obj_ids)

# Summarize observation MJDs for each diaObjectId.
partials = []
for batch in dataset.to_batches(
    columns=needed_cols,
    filter=ds.field('diaObjectId').isin(list(obj_id_set)),
):
    df = batch.to_pandas()
    df['expMidptMJD'] = df['visit'].map(visit_mjd)
    g = df.groupby('diaObjectId')['expMidptMJD'].agg(
        min='min',
        max='max',
    )
    partials.append(g)
combined = pd.concat(partials).groupby('diaObjectId').agg(
    min=('min', 'min'),max=('max', 'max'),
)
obs_mjd_summary = (
    combined[['min', 'max']].reset_index().rename(
        columns={'min': 'obs_mjd_min', 'max': 'obs_mjd_max'})
)

# Summarize detection MJDs for each diaObjectId (S/N > 5).
partials = []
for batch in dataset.to_batches(
    columns=needed_cols,
    filter=(
        ds.field('diaObjectId').isin(list(obj_id_set))
        & ((ds.field('psfFlux') / ds.field('psfFluxErr')) > 5)
    ),
):
    df = batch.to_pandas()
    df['expMidptMJD'] = df['visit'].map(visit_mjd)
    g = df.groupby('diaObjectId')['expMidptMJD'].agg(min='min', max='max')
    partials.append(g)
combined = pd.concat(partials).groupby('diaObjectId').agg(
    min=('min', 'min'), max=('max', 'max')
)
dets_mjd_summary = combined[['min', 'max']].reset_index().rename(
    columns={'min': 'det_mjd_min', 'max': 'det_mjd_max'}
)
mjd_summary = obs_mjd_summary.merge(dets_mjd_summary, on='diaObjectId', how='outer')

# Joining to the main diaobject table to enable filtering.
diaobject = pd.merge(
    diaobject,
    mjd_summary,
    on='diaObjectId',
    how='outer'
)

Now let's write our very strict filter which will include:
- A cut on short duration transients (less than 14 days) but if the observations were taken over >14 days we need to account for that too.
- A Galactic latitude of |b| > 15
- A minimum of 5 DIA sources (detections) in r-band.
- Detections in at least 3 bands.
- A peak SNR cut of >10.
- A fade rate of >30% per day.

In [ ]:
duration_cut = (
    (
        (diaobject['det_mjd_max'] - diaobject['det_mjd_min'] < 14.0)
        & (diaobject['obs_mjd_max'] - diaobject['obs_mjd_min'] > 14.0)
    )
    | (
        (diaobject['det_mjd_max'] - diaobject['det_mjd_min']
         < diaobject['obs_mjd_max'] - diaobject['obs_mjd_min'] - 1.0)
        & (diaobject['obs_mjd_max'] - diaobject['obs_mjd_min'] < 14.0)
    )
)

# Galactic latitude cut
coords = SkyCoord(ra=diaobject['ra'].values*u.deg, dec=diaobject['dec'].values*u.deg)
gal_lat_cut = np.abs(coords.galactic.b.deg) > 15

# At least five r-band detections
ndet_cut = diaobject['r_psfFluxNdata'] >= 5

# Detections in at least three filters
filters = ['u', 'g', 'r', 'i', 'z', 'y']
ndata_columns = [f'{filt}_psfFluxNdata' for filt in filters]
bands_cut = diaobject[ndata_columns].gt(0).sum(axis=1) >= 3

snr_cut = (
    np.abs(diaobject['r_psfFluxMax'])
    / diaobject['r_psfFluxErrMean']
) > 10

faderate_cut = (
    diaobject['r_psfFluxMax']
    / diaobject['r_psfFluxMaxSlope']
) > 0.3

candidates = diaobject[
    duration_cut
    & ndet_cut
    & bands_cut
    & snr_cut
    & faderate_cut
    & gal_lat_cut
].reset_index(drop=True)

candidates

### Binning
Rubin will often have many exposures taken on a single night (especially in the deep drilling fields).
This is good for finding really fast things (e.g. flare stars) but isn't ideal for things like supernovae and makes filtering more difficult.
So here we'll bin the data on a nightly basis for each band.

In [ ]:
def _assign_groups_grouped(times, boundary_change, window_days):
    """
    boundary_change[i] = True if row i starts a new (diaObjectId, band) group,
    or i == 0. Group id resets (increments) at every boundary_change AND
    every time-gap break.
    """
    n = len(times)
    group_id = np.empty(n, dtype=np.int64)
    gid = -1
    anchor = times[0]

    for i in range(n):
        if boundary_change[i]:
            gid += 1
            anchor = times[i]
        elif times[i] - anchor > window_days:
            gid += 1
            anchor = times[i]
        group_id[i] = gid

    return group_id


def _group_by_time_window(df, time_col='mjd', obj_col='diaObjectId', band_col='band', window_hours=12):
    """
    Group rows within `window_hours` of an anchor time, separately per
    (obj_col, band_col) combination.

    Returns a copy of df sorted by [obj_col, band_col, time_col] with an
    added 'group_id' column (globally unique across all objects/bands —
    safe to use directly in groupby, no need to also group by obj_col/band_col).
    """
    window_days = window_hours / 24.0

    out = df.sort_values([obj_col, band_col, time_col]).reset_index(drop=True).copy()

    times = out[time_col].to_numpy()
    objs = out[obj_col].to_numpy()
    bands = out[band_col].to_numpy()

    boundary_change = np.empty(len(objs), dtype=np.bool_)
    boundary_change[0] = True
    boundary_change[1:] = (objs[1:] != objs[:-1]) | (bands[1:] != bands[:-1])

    out['group_id'] = _assign_groups_grouped(times, boundary_change, window_days)
    return out

def bin_data(df, window_hours=12, flux_col='psfFlux'):
    df = _group_by_time_window(df, time_col='expMidptMJD', obj_col='diaObjectId', band_col='band', window_hours=window_hours)
    df_binned = (
        df.groupby('group_id').agg(
            mjd_mean=('expMidptMJD', 'mean'),
            mjd_min=('expMidptMJD', 'min'),
            mjd_max=('expMidptMJD', 'max'),
            flux_mean=(flux_col, 'mean'),
            n=(flux_col, 'size'),
        )
    )
    # then merge in the conditional flux_err separately since agg can't do the conditional logic
    std = df.groupby('group_id')[flux_col].std(ddof=1)
    sem = std / np.sqrt(df_binned['n'])
    first_err = df.groupby('group_id')[flux_col + 'Err'].first()  # any row's err works when n==1
    df_binned['flux_err'] = np.where(df_binned['n'] == 1, first_err, sem)
    df_binned['diaObjectId'] = df.groupby('group_id')['diaObjectId'].first()
    df_binned['band'] = df.groupby('group_id')['band'].first()

    df_binned = df_binned.rename(columns={
        'flux_mean': flux_col,
        'flux_err': flux_col + 'Err',
        'mjd_mean': 'expMidptMJD',
    })

    return df_binned

dataset = ds.dataset(forced_path, format='parquet')
forced_lcs = dataset.to_table(
    filter=ds.field('diaObjectId').isin(candidates['diaObjectId'])
).to_pandas()

# Add MJDs and filters
forced_lcs = pd.merge(
    forced_lcs,
    visit[['visit', 'expMidptMJD']],
    on='visit',
    how='left'
)
forced_lcs_binned = bin_data(forced_lcs, window_hours=12, flux_col='psfFlux')

## Cross-matching
Here we'll cross match with other surveys/catalogues

In [ ]:
from astroquery.vizier import Vizier
from dl import queryClient as qc
from dl.helpers.utils import convert
import time

# Gaia DR3
def gaia_xmatch(df, ra_col="ra", dec_col="dec", id_col="diaObjectId", radius_arcsec=1.0):
    """
    Crossmatch with Gaia catalog via Vizier.
    """
    keep_arr = []
    for _, row in df.iterrows():
        ra = row[ra_col]
        dec = row[dec_col]
        cand_id = row[id_col]

        result = Vizier.query_region(f"{ra} {dec}", radius=f"{radius_arcsec}s", catalog="I/355/gaiadr3")
        if result:
            gaia_data = result[0].to_pandas()
            gaia_data["cand_id"] = cand_id
            keep_arr.append(gaia_data)

    if not keep_arr:
        print("No Gaia matches found.")
        return df

    return df.merge(pd.concat(keep_arr, ignore_index=True), left_on=id_col, right_on="cand_id", how="left")

# Legacy Survey (DECam)
def legacysurvey_xmatch(df, ra_col="ra", dec_col="dec", id_col="diaObjectId", radius_arcsec=1.0, data_release="ls_dr11"):
    """
    Bulk crossmatch via NOIRLab Astro Data Lab (q3c spatial join).
    Requires: pip install datalab-client ; datalab login (once, interactively).
    """
    keep_arr = []
    for _,row in df.iterrows():
        ra = row[ra_col]
        dec = row[dec_col]
        cand_id = row[id_col]

        query = f"""
        SELECT t.ref_cat AS ls_type,
            t.ra, t.dec,
            t.mag_g, t.mag_r, t.mag_i, t.mag_z
        FROM {data_release}.tractor AS t
        WHERE q3c_radial_query(t.ra, t.dec, {ra}, {dec}, {radius_arcsec} / 3600.0)
        """
        try:
            result = qc.query(sql=query, fmt="pandas")
        except Exception as e:
            print(f"Query failed: {e}, retrying in 5 seconds...")
            time.sleep(5)
            try:
                result = qc.query(sql=query, fmt="pandas")
            except Exception as e:
                print(f"Query failed again: {e}")
                continue
        result = convert(result, outfmt="pandas") if not isinstance(result, pd.DataFrame) else result
        result['cand_id'] = cand_id
        # Keep only the nearest Tractor match per candidate.
        #result = result.sort_values("sep_arcsec").drop_duplicates("cand_id", keep="first")
        keep = result[["cand_id", "ls_type", "mag_g", "mag_r", "mag_i", "mag_z"]]
        keep = keep.rename(columns={"cand_id": id_col, "sep_arcsec": "ls_sep_arcsec"})
        keep_arr.append(keep)

    return df.merge(pd.concat(keep_arr, ignore_index=True), on=id_col, how="left")

candidates = legacysurvey_xmatch(candidates, radius_arcsec=5.0, data_release="ls_dr11")
candidates = gaia_xmatch(candidates, radius_arcsec=1.0)
candidates[candidates['ls_type'] != "PSF"]

No Gaia matches but it there are some candidates with non-PSF background sources. So now let's look at the light curves for those: 

In [ ]:
for cand_id in candidates[candidates['ls_type'] != "PSF"]['diaObjectId']:
    forced_lcs_cand = forced_lcs[forced_lcs['diaObjectId'] == cand_id].reset_index(drop=True)

    forced_lcs_cand['mag'] = (np.array(forced_lcs_cand['psfFlux']) * u.nJy).to(u.ABmag).value
    forced_lcs_cand['mag_err'] = 2.5 * np.log10(np.e) * np.abs(forced_lcs_cand['psfFluxErr']/forced_lcs_cand['psfFlux']) # approximate

    ra, dec = diaobject[diaobject['diaObjectId'] == cand_id][['ra', 'dec']].values[0]
    coords = SkyCoord(ra=ra*u.deg, dec=dec*u.deg) 

    forced_lcs_cand['mag'] = correct_extinction(
        forced_lcs_cand['mag'],
        forced_lcs_cand['band'],
        coords,
    )

    plot_mag_lc(forced_lcs_cand)

### ZTF cross-matching
These light curves are pretty sparse so its hard to tell exactly what they are.  Let's try to see if there are data points from the Zwicky Transient Facility.

You'll need `babamul` credentials for this to work.  You can make an account here: https://babamul.caltech.edu/

In [ ]:
import babamul
candidates['ztf_id'] = ''
for cand_id, ra, dec in candidates[['diaObjectId', 'ra', 'dec']].values:
    alerts = babamul.get_alerts(
        'ZTF',
        ra=ra,
        dec=dec,
        radius_arcsec=1.
    )
    if len(alerts) > 0:
        candidates.loc[candidates['diaObjectId'] == cand_id, 'ztf_id'] = alerts[0].objectId
    print(f"Found {len(alerts)} alerts for {int(cand_id)}" + (f" with name {alerts[0].objectId}" if len(alerts) > 0 else ""))

We have some matches! Let's plot their joint light curves.

In [ ]:
from astropy.time import Time

def plot_ztflsst_lc(lsst_lc,ztf_lc):
    for band, color in FILTER_COLORS.items():
        band_lc = lsst_lc[(lsst_lc['band'] == band) & (lsst_lc['mag_err'] < 0.5)]
        if len(band_lc) > 0:
            plt.errorbar(
                band_lc['expMidptMJD'],
                band_lc['mag'],
                yerr=band_lc['mag_err'],
                fmt='o',
                color=color,
                label='LSST ' + band,
                capsize=3,
                markeredgecolor='k',
                markerfacecolor=color,
                alpha=0.7,
            )
        if band in ['g', 'r', 'i']:
            band_lc_ztf = ztf_lc[(ztf_lc['band'] == band) & (ztf_lc['mag_err'] < 0.5)]
            if len(band_lc_ztf) == 0:
                continue
            plt.errorbar(
                band_lc_ztf['mjd'],
                band_lc_ztf['mag'],
                yerr=band_lc_ztf['mag_err'],
                fmt='s',
                color=color,
                label=f"ZTF {band}",
                capsize=3,
                markeredgecolor='k',
                markerfacecolor=color,
                alpha=0.7,
            )
    plt.xlabel('MJD')
    plt.ylabel(r'AB Magnitude')
    plt.gca().invert_yaxis()
    plt.legend()
    plt.show()

for _,row in candidates[candidates['ztf_id'] != ''].iterrows():
    phot = babamul.get_object('ZTF', row['ztf_id']).get_photometry(deduplicated=True)
    ztf_mag = [p.magpsf for p in phot]
    ztf_mag_err = [p.sigmapsf for p in phot]
    ztf_mjd = [Time(p.jd, format='jd').mjd for p in phot]
    ztf_bands = [p.band[-1] for p in phot]

    df = pd.DataFrame({
        'ztf_id': row['ztf_id'],
        'diaObjectId': row['diaObjectId'],
        'mjd': ztf_mjd,
        'mag': ztf_mag,
        'mag_err': ztf_mag_err,
        'band': ztf_bands,
    })

    forced_lcs_cand = forced_lcs[forced_lcs['diaObjectId'] == row['diaObjectId']].reset_index(drop=True)

    forced_lcs_cand['mag'] = (np.array(forced_lcs_cand['psfFlux']) * u.nJy).to(u.ABmag).value
    forced_lcs_cand['mag_err'] = 2.5 * np.log10(np.e) * np.abs(forced_lcs_cand['psfFluxErr']/forced_lcs_cand['psfFlux']) # approximate

    plot_ztflsst_lc(forced_lcs_cand, df)
    

## Activity
From here you can choose your own adventure (in groups preferably to maximise science!). Some ideas:
- Relaxing, changing or creating a new filter and creating scanning pages to identify/classify transients.
- Cross-matching other surveys e.g. ASKAP, DECam, VLA, TNS, ...
- Using the RSP to get more/different classes of object.
- Write some tools for cross-matching, filtering, etc. and push to this github repo.
- ...
> **Tip:** If you find a previously unknown transient, report it to TNS! 